In [ ]:
#==================================
#LIBRARY IMPORTS
#==================================

import pandas as pd   
# -- 1). For creating, reading, and processing tabular data using DataFrames
from sentence_transformers import SentenceTransformer, CrossEncoder 
# -- 1). SentenceTransformer is used to generate semantic embeddings.
# -- 2). CrossEncoder is used for more accurate pairwise similarity scoring or re-ranking.
import warnings
warnings.filterwarnings("ignore")
# -- 1). Suppresses non-critical warning messages to keep the output clean
from tkinter import Tk 
# -- 1).Used to initialize a basic Tkinter GUI window
from tkinter.filedialog import askopenfilename 
# -- 1). Opens a GUI file picker dialog to allow the user to select a file
import os 
# -- 1). Provides utilities for working with file paths, folders, and operating system operations
import torch 
# -- 1). Used for tensor operations and GPU acceleration, if a compatible GPU is available
from transformers import AutoTokenizer 
# -- 1). Loads a tokenizer for converting text into model-compatible tokens
import numpy as np 
# -- 1). Provides support for numerical operations, arrays, and mathematical computations
import sys 
# -- 1). Provides access to system-specific functions, such as exiting the script
import logging 
# -- 1). Used to record logs for debugging, monitoring, and tracking script execution

#=================================
# OWN FUNCTIONS
#=================================

# importing the classes as a package from the functions.py file in the Packages directory
from Packages.functions import all_functions
smart_title_matching_functions=all_functions()

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

"""                   -- Custom logger configuration. --
    This logger is created using the current module name, configured with an
    INFO log level, and attached to a file handler that writes log messages
    to 'Smart.log'. The log file is opened in write mode, so it is overwritten
    each time the script starts.
"""
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
f = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh = logging.FileHandler('Smart.log', mode='w')   # ✅ overwrite file
fh.setFormatter(f)
logger.addHandler(fh)

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

"""
This block captures the IP category selected by the user and validates the input.

The user is prompted to enter one of the allowed IP category values:
    - S   : Standalone
    - SE  : Series
    - ALL : Series, Season, and Episodic

The entered value is trimmed, converted to uppercase, and checked against the
allowed values. If the user enters any value other than S, SE, or ALL, the
script raises a ValueError.

The script also defines predefined IP type lists for different processing modes:
    - Series
    - Standalone
    - Seasons
    - Episodic
    - Series_Season_Episode

These lists are used later to filter or classify records based on their IP type.

Finally, the Tkinter root window is hidden using `Tk().withdraw()` so that only
file picker dialogs are shown to the user without displaying an extra blank GUI
window.

"""
IP = input("Enter the IP_Category [SE-Series, S-Standalone, ALL-Series_Season_Episodic]: ").strip().upper()
valid_ip_values = {"S", "SE", "ALL"}
if IP not in valid_ip_values:
    raise ValueError("Invalid IP value. Please enter only S, SE, or ALL.")

Series= ["Series",'Mini Series','Feature','Special','Short','TV Movie','Made For Video']
Standalone=['Special','Feature | Special','Feature | TV Movie','Short','TV Movie','Feature','Made For Video','Feature | Short','Supplemental','']
Seasons=['Season','Supplemental','Non-IP','Term Deal','Ancillary/Derivative','Consumer Products','Live Stage','Mini Series','Special']
Episodic=['Episode','Special','Short','Segment','Music','Publishing','Pilot','Game','']
Series_Season_Episode=["Series",'Mini Series','Feature','Special','Short','TV Movie','Made For Video','Season','Supplemental','Non-IP','Term Deal','Ancillary/Derivative','Consumer Products','Live Stage','Mini Series','Episode','Special','Short','Segment','Music','Publishing','Pilot','Game','']

# Hide the root window
Tk().withdraw()

# Open file picker dialog
file_path = askopenfilename(title="Select Excel file", filetypes=[("Excel files", "*.xlsx")])

# Load CSV into DataFrame
try:
    if file_path:
        INPUT_TITLES = pd.read_excel(file_path)
        INPUT_TITLES_0=INPUT_TITLES.copy()
    else:
        logger.info("No file selected.")
        sys.exit(0)
except Exception as e:
    logger.error(f"Error loading file: {e}")
    sys.exit(1)

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Reset index and create Sno

"""
This block prepares the input titles based on the selected IP category.

For `S` and `SE` processing modes, the `Features` column is converted to text,
split into multiple title variants using `split_aka`, expanded into separate
rows using `explode`, and then reset with a clean index.

For `ALL` processing mode, the script prepares series, season, and episodic
titles for matching. It identifies the correct episode title column, ensures
required columns such as `SERIES_TITLE` and episode title exist, and fills
missing values with blank strings.

The script then creates a `MATCH_TITLE_RAW` column based on the IP type:
    - Series records use `SERIES_TITLE`.
    - Season records use the episode title column.
    - Episodic records use `genric_merger` to combine episode title, series
    title, season number, and episode number into an AKA-style match title.

Before splitting AKA titles, the original input data is preserved in
`INPUT_TITLES_0`. Then `MATCH_TITLE_RAW` is split into multiple searchable
title variants using `split_aka`, expanded into separate rows, and the index is
reset.

After title preparation, the script normalizes the matching title column using
`normalize_title`. It also escapes single quotes by replacing `'` with `''` so
the values can be safely inserted into a SQL query string.

Finally, the script builds `Input_titles`, a comma-separated string of tuples
containing:
    - Sno
    - normalized title
    - year value

This `Input_titles` string is later used in SQL-based matching logic.

"""
if IP in ("S", "SE"):
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].astype(str)
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].apply(smart_title_matching_functions.split_aka)
    INPUT_TITLES = INPUT_TITLES.explode("Features").reset_index(drop=True)

elif IP == "ALL":
    episode_col = "EPISODE_TITLE" if "EPISODE_TITLE" in INPUT_TITLES.columns else "Episode"

    if "SERIES_TITLE" not in INPUT_TITLES.columns:
        INPUT_TITLES["SERIES_TITLE"] = ""

    if episode_col not in INPUT_TITLES.columns:
        INPUT_TITLES[episode_col] = ""

    INPUT_TITLES["SERIES_TITLE"] = INPUT_TITLES["SERIES_TITLE"].fillna("").astype(str)
    INPUT_TITLES[episode_col] = INPUT_TITLES[episode_col].fillna("").astype(str)

    INPUT_TITLES["MATCH_TITLE_RAW"] = ""
    series_mask  = INPUT_TITLES["IP_TYPE"].isin(Series)
    season_mask  = INPUT_TITLES["IP_TYPE"].isin(Seasons)
    episode_mask = INPUT_TITLES["IP_TYPE"].isin(Episodic)


    INPUT_TITLES.loc[series_mask, "MATCH_TITLE_RAW"] = INPUT_TITLES.loc[
        series_mask, "SERIES_TITLE"
    ]

    INPUT_TITLES.loc[season_mask, "MATCH_TITLE_RAW"] = INPUT_TITLES.loc[
        season_mask, "EPISODE_TITLE"
    ]

    INPUT_TITLES.loc[episode_mask, "MATCH_TITLE_RAW"] = INPUT_TITLES.loc[
        episode_mask
    ].apply(
        lambda row: smart_title_matching_functions.genric_merger(
            row[episode_col],
            row["SERIES_TITLE"],
            row.get("SeasonNumber"),
            row.get("EpisodeNumber")
        ),
        axis=1
    )

    # Important: preserve raw title in INPUT_TITLES_0 before split/explode
    INPUT_TITLES_0 = INPUT_TITLES.copy()

    INPUT_TITLES["MATCH_TITLE"] = INPUT_TITLES["MATCH_TITLE_RAW"].apply(
        smart_title_matching_functions.split_aka
    )
    INPUT_TITLES = INPUT_TITLES.explode("MATCH_TITLE").reset_index(drop=True)

## Normalize titles and escape single quotes for SQL
if IP in ("S","SE"):
    INPUT_TITLES["Features"]=INPUT_TITLES["Features"].apply(smart_title_matching_functions.normalize_title)
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].str.replace("'", "''")

    Input_titles = ",".join([
        f"('{i}', '{j}', {int(k)})" if not pd.isna(k) else f"('{i}', '{j}', {0})"
        for i, j, k in zip(
            INPUT_TITLES["Sno"],
            INPUT_TITLES["Features"],
            INPUT_TITLES["Years"]
        )
    ])
elif IP == "ALL":
    INPUT_TITLES["MATCH_TITLE"] = INPUT_TITLES["MATCH_TITLE"].apply(smart_title_matching_functions.normalize_title)
    INPUT_TITLES["MATCH_TITLE"] = INPUT_TITLES["MATCH_TITLE"].str.replace("'", "''")

    Input_titles = ",".join([
        f"('{i}', '{j}', {int(k)})" if not pd.isna(k) else f"('{i}', '{j}', {0})"
        for i, j, k in zip(
            INPUT_TITLES["Sno"],
            INPUT_TITLES["MATCH_TITLE"],
            INPUT_TITLES["Years"]
        )
    ])
# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
## Filtered data from snowflake and check all the Pid's present:

# Create connection
conn = smart_title_matching_functions.get_connection()

# Create a cursor
cur = conn.cursor()


# Convert list to SQL-friendly string
if IP=="SE":
    IP_Category = ",".join(f"'{pid}'" for pid in Series)
elif IP=="S":
    IP_Category = ",".join(f"'{pid}'" for pid in Standalone)
elif IP=="ALL":
    IP_Category = ",".join(f"'{pid}'" for pid in Series_Season_Episode)

#MPM_Episode_str = ",".join(f"'{pid}'" for pid in Episode)
if IP == "ALL":
    IS_PARENTS = "'P','C'"
elif IP == "SE":
    IS_PARENTS = "'P','C'"
else:
    IS_PARENTS = "'C'"

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
---------- SQL QUERY FOR ATOM TITLE MATCHING (Simple Explanation) ----------

This SQL query retrieves and ranks possible ATOM title matches for the prepared
input titles.

The query first builds hierarchy information from ATOM episodic (BOLT_MSC_CDS_PROD.ATOM_BI.EPISODIC_TITLE_HIERARCHY_VW) and title (BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE)  tables.
It identifies series, season, episode, parent, and child relationships, fills
missing hierarchy identifiers where possible, and creates hierarchy-based title
values for generic episode or season titles.

The input titles are passed into the query through the `Input_titles` VALUES
clause. Each input title is split into individual words, and the query compares
those words against title words extracted from multiple ATOM title sources:
    - TITLE
    - AKA_PKA_TITLES
    - LIBRARY_TITLE_FULL or LIBRARY_TITLE_SHORT
    - Hierarchy-generated titles

The query then identifies candidate matches based on shared title words. A
candidate is kept when it matches at least 20 percent of the input title words.

Year validation is applied using the input year and ATOM release or production
year. Matches are retained when the year is missing, the input year is zero, or
the ATOM year is within a plus or minus 3 year range from the input year.

For `ALL` mode, the query also applies generic episode logic so that generic
episode-like input titles are matched only against generic episode-like ATOM
titles.

The matched candidates are enriched with additional metadata from ATOM, including:
    - Node identifier
    - IP type
    - MPM number
    - Product and family identifiers
    - PI UUID
    - Property, HBO, Meta, Turner, MMS3, DASH, Aleph, and iBroadcast IDs
    - IMDb TT code
    - Parent entity and parent title
    - Children count summary
    - Parent or child indicator

Records marked as SDS or DO NOT USE are removed from the final candidate set.

Finally, the query ranks candidates for each input ID using:
    - Jaro-Winkler similarity, sorted descending
    - Edit distance, sorted ascending

The final output returns the top ranked candidates per input ID, limited to rows
with Jaro-Winkler similarity of 50 or higher and rank less than or equal to 30.
Each row is labeled as `Match` or `No match` based on whether a NODE_IDENTIFIER
was found.

"""
# First query
logger.info("Loading snowflake data..")
query = fr"""
WITH ATOM_DATA AS (
    SELECT 
        E.SERIES_IDENTIFIER,
        E.NODE_IDENTIFIER,
        E.IP_TYPE,
        E.MPM_NUMBER,
        E.SEASON_MPM_NUMBER,
        E.SERIES_MPM_NUMBER
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.EPISODIC_TITLE_HIERARCHY_VW E
),

HIERARCHY_INFO1 AS (
    SELECT
        E.SERIES_IDENTIFIER,
        E.NODE_IDENTIFIER,
        E.IP_TYPE,
        COALESCE(E.MPM_NUMBER, T.MPM_NUMBER) AS EPISODE_MPM_NUMBER,
        COALESCE(E.SEASON_MPM_NUMBER, T.MPM_PRODUCT_NUMBER) AS SEASON_MPM_NUMBER,
        COALESCE(E.SERIES_MPM_NUMBER, T.MPM_FAMILY_NUMBER) AS SERIES_MPM_NUMBER,
        T.PROPERTY_ID,
        T.PI_UUID
    FROM ATOM_DATA E
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON E.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

SERIES_LOOKUP AS (
    SELECT
        SERIES_MPM_NUMBER,
        MAX(SERIES_IDENTIFIER) AS SERIES_IDENTIFIER
    FROM HIERARCHY_INFO1
    GROUP BY SERIES_MPM_NUMBER
),

FILLING_IDENTIFIERS AS (
    SELECT
        SL.SERIES_MPM_NUMBER,
        T.NODE_IDENTIFIER AS SERIES_IDENTIFIER
    FROM SERIES_LOOKUP SL
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON SL.SERIES_MPM_NUMBER = T.MPM_NUMBER
    WHERE T.NODE_IDENTIFIER IS NOT NULL
),

HIERARCHY_INFO_F AS (
    SELECT
        COALESCE(H.SERIES_IDENTIFIER, SL.SERIES_IDENTIFIER, SSL.SERIES_IDENTIFIER) AS SERIES_IDENTIFIER,
        SSL.SEASON_IDENTIFIER,
        H.NODE_IDENTIFIER,
        H.IP_TYPE,
        H.EPISODE_MPM_NUMBER,
        H.SEASON_MPM_NUMBER,
        COALESCE(H.SERIES_MPM_NUMBER, SSL.SERIES_MPM_NUMBER) AS SERIES_MPM_NUMBER,
        H.PROPERTY_ID,
        H.PI_UUID
    FROM HIERARCHY_INFO1 H
    LEFT JOIN FILLING_IDENTIFIERS SL
        ON H.SERIES_MPM_NUMBER = SL.SERIES_MPM_NUMBER
    LEFT JOIN (
        SELECT
            SEASON_IDENTIFIER,
            SEASON_MPM_NUMBER,
            SERIES_IDENTIFIER,
            SERIES_MPM_NUMBER
        FROM (
            SELECT
                H2.SEASON_MPM_NUMBER,
                T.NODE_IDENTIFIER AS SEASON_IDENTIFIER,
                T.MPM_FAMILY_NUMBER AS SERIES_MPM_NUMBER,
                T.NODE_IDENTIFIER AS SERIES_IDENTIFIER,
                ROW_NUMBER() OVER (
                    PARTITION BY H2.SEASON_MPM_NUMBER
                    ORDER BY T.NODE_IDENTIFIER
                ) AS RN
            FROM HIERARCHY_INFO1 H2
            LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
                ON H2.SEASON_MPM_NUMBER = T.MPM_NUMBER
        )
        WHERE RN = 1
    ) SSL
        ON H.SEASON_MPM_NUMBER = SSL.SEASON_MPM_NUMBER
),

Heirachy_Integrity1 AS (
    SELECT 
        CASE 
            WHEN HF.NODE_IDENTIFIER = HF.SERIES_IDENTIFIER THEN NULL 
            ELSE HF.SERIES_IDENTIFIER 
        END AS SERIES_IDENTIFIER,
        HF.SEASON_IDENTIFIER,
        HF.NODE_IDENTIFIER,
        HF.IP_TYPE,
        T.MPM_FAMILY_NUMBER,
        T.MPM_PRODUCT_NUMBER,
        T.MPM_NUMBER
    FROM HIERARCHY_INFO_F HF
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON HF.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity1_series AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        COALESCE(T.TITLE, T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT) AS SERIES_TITLE
    FROM Heirachy_Integrity1 H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.SERIES_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity1_season AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        H1.SERIES_TITLE,
        COALESCE(T.TITLE, T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT) AS SEASON_TITLE
    FROM Heirachy_Integrity1_series H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.SEASON_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity_episode AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        H1.SERIES_TITLE, H1.SEASON_TITLE,
        COALESCE(T.TITLE, T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT) AS EPISODE_TITLE
    FROM Heirachy_Integrity1_season H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity AS (
    SELECT *,
    CASE
        WHEN IP_TYPE IN ('Episode','Special','Short','Segment','Music','Publishing','Pilot','Game','') AND REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
            '^(S\\d+\\s*)?(EPISODE|EPI|EP|E)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SEASON_TITLE, SERIES_TITLE) || ' ' || EPISODE_TITLE

        WHEN IP_TYPE IN ('Season','Supplemental','Non-IP','Term Deal','Ancillary/Derivative','Consumer Products','Live Stage') AND REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
            '^(SEASON|SEA|S)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SERIES_TITLE, SEASON_TITLE) || ' ' || EPISODE_TITLE

        WHEN REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
            '^(S\\d+\\s*)?(EPISODE|EPI|EP|E)\\s*#?\\s*\\d+$|^(SEASON|SEA|S)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SERIES_TITLE, SEASON_TITLE) || ' ' || EPISODE_TITLE

        ELSE EPISODE_TITLE
    END AS Titles_heirachy
    FROM Heirachy_Integrity_episode
),

/* ATOM Search (optimized — single D_TITLE scan): */
raw_inputs AS (
    SELECT 
        column1 AS id,
        column2 AS input_title,
        column3 AS input_year,
        CASE
            WHEN REGEXP_LIKE(
                UPPER(REGEXP_REPLACE(column2, '[^A-Z0-9 ]', ' ')),
                '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
            )
            THEN 1
            ELSE 0
        END AS INPUT_IS_GENERIC_EPISODE
    FROM VALUES 
        {Input_titles}
),
input_words AS (
    SELECT DISTINCT
        r.id,
        r.input_title,
        r.input_year,
        r.INPUT_IS_GENERIC_EPISODE,
        TRIM(f.value::string) AS input_word
    FROM raw_inputs r,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(r.input_title), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE TRIM(f.value::string) <> ''
),

input_word_count AS (
    SELECT 
        id,
        COUNT(*) AS total_input_words
    FROM input_words
    GROUP BY id
),

title_base AS (

    /* Primary: TITLE */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        m.TITLE AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word,
        CASE
            WHEN REGEXP_LIKE(
                UPPER(REGEXP_REPLACE(COALESCE(m.TITLE, ''), '[^A-Z0-9 ]', ' ')),
                '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
            )
            THEN 1
            ELSE 0
        END AS ATOM_IS_GENERIC_EPISODE
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(m.TITLE), '[^[:alnum:] ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
    AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Secondary: AKA_PKA_TITLES */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        m.AKA_PKA_TITLES AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word,
        CASE
            WHEN REGEXP_LIKE(
                UPPER(REGEXP_REPLACE(COALESCE(m.AKA_PKA_TITLES, ''), '[^A-Z0-9 ]', ' ')),
                '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
            )
            THEN 1
            ELSE 0
        END AS ATOM_IS_GENERIC_EPISODE
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(m.AKA_PKA_TITLES), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
    AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Third: LIBRARY_TITLE_FULL */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        COALESCE(m.LIBRARY_TITLE_FULL, m.library_title_short) AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word,
        CASE
            WHEN REGEXP_LIKE(
                UPPER(REGEXP_REPLACE(COALESCE(COALESCE(m.LIBRARY_TITLE_FULL, m.library_title_short), ''), '[^A-Z0-9 ]', ' ')),
                '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
            )
            THEN 1
            ELSE 0
        END AS ATOM_IS_GENERIC_EPISODE
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(COALESCE(m.LIBRARY_TITLE_FULL, m.library_title_short)), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
    AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Fourth: Generic titles */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        H.Titles_heirachy AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word,
        CASE
            WHEN REGEXP_LIKE(
                UPPER(REGEXP_REPLACE(COALESCE(H.Titles_heirachy, ''), '[^A-Z0-9 ]', ' ')),
                '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
            )
            THEN 1
            ELSE 0
        END AS ATOM_IS_GENERIC_EPISODE
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m
    LEFT JOIN Heirachy_Integrity H
        ON m.NODE_IDENTIFIER = H.NODE_IDENTIFIER,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(H.Titles_heirachy), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
    AND TRIM(f.value::string) <> ''
),

title_words AS (
    SELECT DISTINCT
        NODE_IDENTIFIER,
        LIBRARY_TITLE,
        PI_UUID,
        MPM_NUMBER,
        IP_TYPE,
        YEARS,
        title_word,
        ATOM_IS_GENERIC_EPISODE
    FROM title_base
),
matched_titles AS (
    SELECT 
        i.id,
        t.NODE_IDENTIFIER,
        t.LIBRARY_TITLE,
        t.PI_UUID,
        t.MPM_NUMBER,
        t.IP_TYPE,
        t.YEARS,
        i.input_title,
        i.input_year,
        MAX(COALESCE(r.INPUT_IS_GENERIC_EPISODE, 0)) AS INPUT_IS_GENERIC_EPISODE,
        MAX(COALESCE(t.ATOM_IS_GENERIC_EPISODE, 0)) AS ATOM_IS_GENERIC_EPISODE,
        COUNT(DISTINCT t.title_word) AS matched_words
    FROM input_words i
    JOIN raw_inputs r
        ON i.id = r.id
    JOIN title_words t
        ON i.input_word = t.title_word
    GROUP BY 
        i.id,
        t.NODE_IDENTIFIER,
        t.LIBRARY_TITLE,
        t.PI_UUID,
        t.MPM_NUMBER,
        t.IP_TYPE,
        t.YEARS,
        i.input_title,
        i.input_year
),

Final_matches AS (
    SELECT 
        m.id,
        m.input_title AS Input_title,
        m.input_year,
        m.LIBRARY_TITLE AS Atom_title,
        m.YEARS,
        m.IP_TYPE,
        m.NODE_IDENTIFIER,
        m.MPM_NUMBER,
        m.PI_UUID,
        CASE 
            WHEN m.YEARS IS NULL THEN 'NULL'
            WHEN m.input_year = 0 THEN 'NULL'
            WHEN SPLIT_PART(m.YEARS, '|', 1) BETWEEN m.input_year - 3 AND m.input_year + 3 
                THEN 'GOOD_MATCH'
            ELSE 'CHECK_YEAR'
        END AS YEAR_MATCH_FLAG
    FROM matched_titles m
    JOIN input_word_count iwc
        ON m.id = iwc.id
    WHERE m.matched_words >= CEIL(0.20 * iwc.total_input_words)

    -- ✅ Generic input must match only generic ATOM title
    AND (
            '{IP}' <> 'ALL'
            OR NOT REGEXP_LIKE(
                    UPPER(REGEXP_REPLACE(COALESCE(m.input_title, ''), '[^A-Z0-9 ]', ' ')),
                    '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
                )
            OR REGEXP_LIKE(
                    UPPER(REGEXP_REPLACE(COALESCE(m.LIBRARY_TITLE, ''), '[^A-Z0-9 ]', ' ')),
                    '(^| )((SEASON|SEA|S) *[0-9]+ *(EPISODE|EPI|EP|E) *#? *[0-9]+|S[0-9]+ *E[0-9]+|(EPISODE|EPI|EP|E) *#? *[0-9]+)( |$)'
                )
        )
),


Atom_title_matching AS (
    SELECT *
    FROM Final_matches
    WHERE YEAR_MATCH_FLAG IN ('NULL', 'GOOD_MATCH')
),

FINAL_OUTPUT AS (
    SELECT 
        A.*,
        CASE
            WHEN EXISTS (
                SELECT 1 FROM Heirachy_Integrity H
                WHERE H.SERIES_IDENTIFIER = A.NODE_IDENTIFIER
            ) THEN 'P'
            ELSE 'C'
        END AS IS_PARENT
    FROM Atom_title_matching A
),

Final_matching_titles AS (
    SELECT
        id,
        input_title,
        Atom_title,
        input_year,
        YEARS,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        F.PI_UUID,
        H.SERIES_IDENTIFIER,
        F.IS_PARENT
    FROM FINAL_OUTPUT F
    LEFT JOIN Heirachy_Integrity H
        ON F.NODE_IDENTIFIER = H.NODE_IDENTIFIER
    WHERE IS_PARENT IN ({IS_PARENTS})
),

FINAL_OUTPUT_last1 AS (
    SELECT
        id,
        input_title,
        Atom_title,
        A1.LIBRARY_TITLE_FULL,
        input_year,
        YEARS,
        CASE
        WHEN REGEXP_LIKE(COALESCE(A1.ALTERNATE_IDENTIFIERS_VALUE,''), 'tt[0-9]+', 'i')
        THEN REGEXP_SUBSTR(A1.ALTERNATE_IDENTIFIERS_VALUE, 'tt[^ ]+', 1, 1, 'i')
        END AS TT_CODES,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        A1.MPM_PRODUCT_NUMBER,
        F.PI_UUID,
        A1.PROPERTY_ID,
        A1.HBO_ID,
        A1.META_ID,
        A1.TURNER_TITLEID,
        A1.MMS3_MCode,
        A1.DASH_Title_ID,
        A1.Aleph_ID,
        A1.iBroadcast_EMEA_ID,
        A1.iBroadcast_APAC_ID,
        A.NODE_IDENTIFIER AS PARENT_ENTITY,
        COALESCE(A.LIBRARY_TITLE_FULL, A.LIBRARY_TITLE_SHORT) AS PARENT_TITLE,
        A.MPM_NUMBER AS PARENT_MPM,
        CASE WHEN (UPPER(A1.LIBRARY_TITLE_FULL) LIKE 'SDS%' OR UPPER(A1.LIBRARY_TITLE_FULL) LIKE '%SDS%' OR UPPER(A1.LIBRARY_TITLE_FULL) LIKE '%DO NOT USE%') OR
                (UPPER(A1.LIBRARY_TITLE_SHORT) LIKE 'SDS%' OR UPPER(A1.LIBRARY_TITLE_SHORT) LIKE '%SDS%' OR UPPER(A1.LIBRARY_TITLE_SHORT) LIKE '%DO NOT USE%') OR
                (UPPER(A1.TITLE) LIKE 'SDS%' OR UPPER(A1.TITLE) LIKE '%SDS%' OR UPPER(A1.TITLE) LIKE '%DO NOT USE%') 
            THEN 'Reject'
            ELSE 'Pass' 
        END AS SDS_CHECK_FLAG,
        F.IS_PARENT
    FROM Final_matching_titles F
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE A1
        ON F.NODE_IDENTIFIER = A1.NODE_IDENTIFIER
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE A
        ON F.SERIES_IDENTIFIER = A.NODE_IDENTIFIER
),
FINAL_OUTPUT_last as (
SELECT * 
FROM FINAL_OUTPUT_last1
WHERE SDS_CHECK_FLAG='Pass'
),
children_summary AS (
    SELECT
        SERIES_IDENTIFIER,
        LISTAGG(IP_TYPE || '[' || cnt || ']', ' + ') WITHIN GROUP (ORDER BY IP_TYPE) AS CHILDREN_COUNT
    FROM (
        SELECT SERIES_IDENTIFIER, IP_TYPE, COUNT(*) AS cnt
        FROM Heirachy_Integrity
        GROUP BY SERIES_IDENTIFIER, IP_TYPE
    )
    GROUP BY SERIES_IDENTIFIER
),
Snowflake_final_result AS (
    SELECT 
        r.id,
        r.input_title,
        F.ATOM_TITLE,
        F.LIBRARY_TITLE_FULL,
        r.input_year,
        F.YEARS,
        F.TT_CODES,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        F.MPM_PRODUCT_NUMBER,
        F.PI_UUID,
        F.PROPERTY_ID,
        F.HBO_ID,
        F.META_ID,
        F.TURNER_TITLEID,
        F.MMS3_MCode,
        F.DASH_Title_ID,
        F.Aleph_ID,
        F.iBroadcast_EMEA_ID,
        F.iBroadcast_APAC_ID,
        F.PARENT_ENTITY,
        F.PARENT_TITLE,
        F.PARENT_MPM,
        CS.CHILDREN_COUNT AS CHILDREN_STATUS,
        F.IS_PARENT
    FROM raw_inputs r
    LEFT JOIN FINAL_OUTPUT_last F
        ON r.id = F.id
    LEFT JOIN children_summary CS
        ON F.NODE_IDENTIFIER = CS.SERIES_IDENTIFIER
),

ranked_candidates AS (
    SELECT
        F.*,
        JAROWINKLER_SIMILARITY(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) AS JW,
        EDITDISTANCE(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) AS ED,
        ROW_NUMBER() OVER (
            PARTITION BY F.id
            ORDER BY
                JAROWINKLER_SIMILARITY(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) DESC,
                EDITDISTANCE(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) ASC
        ) AS RN
    FROM Snowflake_final_result F
)

SELECT *,
CASE WHEN NODE_IDENTIFIER IS NULL THEN 'No match' ELSE 'Match' END AS MATCHES
FROM ranked_candidates
WHERE JW >= 50
AND RN <= 30
ORDER BY id, JW DESC, ED ASC;
"""

try:
    logger.info("Executing query on Snowflake...")
    with conn.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
except Exception as e:
    logger.error(f"Error executing query check the VPN (or credentials): {e}")
    sys.exit(1)

SnowFlake_Results = pd.DataFrame(rows, columns=cols)


# Close connection
cur.close()
conn.close()

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
This block separates Snowflake match results based on the selected IP category.

When `IP` is `ALL`, the script separates the original input records into three
groups using `IP_TYPE`:
    - Series records
    - Season records
    - Episode records

It converts the input `Sno` values and Snowflake result `ID` values to strings
so they can be compared consistently.

The Snowflake results are then split into separate DataFrames:
    - `Series_SnowFlake_Results` contains matched Series candidates.
    - `Season_SnowFlake_Results` contains matched Season candidates.
    - `Episode_SnowFlake_Results` contains matched Episodic candidates.

Each output DataFrame is filtered by:
    - Matching input ID
    - Valid IP type list
    - Match status equal to `Match`
    - Parent or child indicator where applicable

For `S` and `SE` modes, the script separates the Snowflake results into:
    - `No_matches`: records where no match was found
    - `Matches`: records where a match was found

The matched records are then converted to numeric nullable integer IDs so they
can be joined, ranked, or processed consistently in later steps.

In the final fallback branch, matched records are also merged with the original
input titles and reordered to keep only the required output columns.
"""

if IP == "ALL":
    # Separating Seasons and Epiosdes
    Series_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"]=="Series"]["Sno"].to_list()]
    Seasons_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"]=="Season"]["Sno"].to_list()]
    Episodes_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"]=="Episode"]["Sno"].to_list()]
    SnowFlake_Results["ID"] = SnowFlake_Results["ID"].astype(str)

    Series_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"].eq("Series")]["Sno"].to_list()]
    Seasons_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"].eq("Season")]["Sno"].to_list()]
    Episodes_id = [str(i) for i in INPUT_TITLES_0[INPUT_TITLES_0["IP_TYPE"].eq("Episode")]["Sno"].to_list()]

    Series_SnowFlake_Results = SnowFlake_Results[
        SnowFlake_Results["ID"].isin(Series_id)
        & SnowFlake_Results["IP_TYPE"].isin(Series)
        & SnowFlake_Results["IS_PARENT"].isin(["P","C"])
        & SnowFlake_Results["MATCHES"].eq("Match")
    ].copy()

    Season_SnowFlake_Results = SnowFlake_Results[
        SnowFlake_Results["ID"].isin(Seasons_id)
        & SnowFlake_Results["IP_TYPE"].isin(Seasons)
        & SnowFlake_Results["MATCHES"].eq("Match")
    ].copy()

    Episode_SnowFlake_Results = SnowFlake_Results[
        SnowFlake_Results["ID"].isin(Episodes_id)
        & SnowFlake_Results["IP_TYPE"].isin(Episodic)
        & SnowFlake_Results["IS_PARENT"].isin(["C"])
        & SnowFlake_Results["MATCHES"].eq("Match")
    ].copy()

elif IP in ("S","SE"):
    No_matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="No match"]
    Matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="Match"]
    Matches["ID"] = pd.to_numeric(Matches["ID"], errors="coerce").astype("Int64")
else:
    No_matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="No match"]
    Matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="Match"]
    Matches["ID"] = pd.to_numeric(Matches["ID"], errors="coerce").astype("Int64")
    Matches=pd.merge(Matches,INPUT_TITLES,how="left",left_on="ID",right_on="Sno")
    Matches=Matches[["ID","SERIES_TITLE","PARENT_TITLE","INPUT_TITLE","ATOM_TITLE","INPUT_YEAR","TT_CODES","YEARS","IP_TYPE","NODE_IDENTIFIER","MPM_NUMBER","MPM_PRODUCT_NUMBER","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID',"PARENT_ENTITY","PARENT_MPM"]]

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## SEMANTIC SCORING:

"""
This block initializes the semantic scoring setup used for title matching.

First, the user selects an output folder where generated files will be saved.
The script then logs that the AI models are being loaded and disables TensorFlow
usage for Transformers by setting `TRANSFORMERS_NO_TF` to `1`.

A SentenceTransformer model, `all-MiniLM-L6-v2`, is loaded to generate semantic
embeddings for title comparison. A CrossEncoder model name is also defined for
more detailed pairwise scoring of borderline matches.

The tokenizer for the CrossEncoder model is loaded to determine the maximum
supported token length. If the tokenizer returns an unusually large or missing
maximum length, the script safely defaults it to 512.

The script then prepares sample title pairs to estimate token lengths:
    - For `S` and `SE`, it uses `INPUT_TITLE` and `ATOM_TITLE` from `Matches`.
    - For `ALL`, it builds token-checking data separately from Series, Season,
    and Episode Snowflake match results.

For `ALL` mode, the script maps input IDs back to their original input titles
from `INPUT_TITLES_0`, using:
    - `SERIES_TITLE` for Series records.
    - `MATCH_TITLE_RAW` for Season and Episode records.

After collecting title pairs, the tokenizer calculates token lengths for each
input-title and candidate-title pair. The 95th percentile of token lengths is
used to choose a practical CrossEncoder maximum length, capped by the model's
maximum supported length.

Finally, the CrossEncoder model is loaded using the selected token length with
a small safety margin. This optimized length helps reduce unnecessary processing
while still supporting most title pairs during cross-encoder scoring.
"""

Output_folder=smart_title_matching_functions.select_output_folder()
# Step 0: Semantic Matching (AI)
logger.info("Loading the Model...")

os.environ["TRANSFORMERS_NO_TF"] = "1"

"""
Note: The HuggingFace website is blocked if you are on the VPN. 
If you are on the VPN, please download the model from HuggingFace and 
give the path here if you want to use a local model instead of downloading it every time.

"""
## Download the model from HuggingFace and give the path here if you want to use a local model instead of downloading it every time.
model = SentenceTransformer("all-MiniLM-L6-v2") 
## Download the model from HuggingFace and give the path here if you want to use a local model instead of downloading it every time.
model_name = "cross-encoder/stsb-roberta-base"  

tokenizer = AutoTokenizer.from_pretrained(model_name)

model_max = tokenizer.model_max_length
if model_max is None or model_max > 10000:
    model_max = 512

token_length_source = pd.DataFrame()

if IP in ("S", "SE"):
    token_length_source = Matches[["INPUT_TITLE", "ATOM_TITLE"]].copy()

elif IP == "ALL":
    token_frames = []

    if not Series_SnowFlake_Results.empty:
        temp = Series_SnowFlake_Results.copy()
        temp["ID_INT"] = pd.to_numeric(temp["ID"], errors="coerce").astype("Int64")
        series_title_map = INPUT_TITLES_0.set_index("Sno")["SERIES_TITLE"]
        temp["INPUT_TITLE_FOR_TOKEN"] = temp["ID_INT"].map(series_title_map)
        token_frames.append(
            temp[["INPUT_TITLE_FOR_TOKEN", "ATOM_TITLE"]].rename(
                columns={"INPUT_TITLE_FOR_TOKEN": "INPUT_TITLE"}
            )
        )

    if not Season_SnowFlake_Results.empty:
        temp = Season_SnowFlake_Results.copy()
        temp["ID_INT"] = pd.to_numeric(temp["ID"], errors="coerce").astype("Int64")
        match_title_map = INPUT_TITLES_0.set_index("Sno")["MATCH_TITLE_RAW"]
        temp["INPUT_TITLE_FOR_TOKEN"] = temp["ID_INT"].map(match_title_map)
        token_frames.append(
            temp[["INPUT_TITLE_FOR_TOKEN", "ATOM_TITLE"]].rename(
                columns={"INPUT_TITLE_FOR_TOKEN": "INPUT_TITLE"}
            )
        )

    if not Episode_SnowFlake_Results.empty:
        temp = Episode_SnowFlake_Results.copy()
        temp["ID_INT"] = pd.to_numeric(temp["ID"], errors="coerce").astype("Int64")
        match_title_map = INPUT_TITLES_0.set_index("Sno")["MATCH_TITLE_RAW"]
        temp["INPUT_TITLE_FOR_TOKEN"] = temp["ID_INT"].map(match_title_map)
        token_frames.append(
            temp[["INPUT_TITLE_FOR_TOKEN", "ATOM_TITLE"]].rename(
                columns={"INPUT_TITLE_FOR_TOKEN": "INPUT_TITLE"}
            )
        )

    if token_frames:
        token_length_source = pd.concat(token_frames, ignore_index=True, sort=False)

token_lengths = []

if not token_length_source.empty:
    token_length_source["INPUT_TITLE"] = token_length_source["INPUT_TITLE"].fillna("").astype(str)
    token_length_source["ATOM_TITLE"] = token_length_source["ATOM_TITLE"].fillna("").astype(str)

    for t1, t2 in zip(
        token_length_source["INPUT_TITLE"].tolist(),
        token_length_source["ATOM_TITLE"].tolist()
    ):
        enc = tokenizer(
            t1,
            t2,
            truncation=True,
            max_length=model_max
        )
        token_lengths.append(len(enc["input_ids"]))

if token_lengths:
    p95 = int(np.percentile(token_lengths, 95))
    chosen_len = min(p95, model_max)
else:
    chosen_len = min(128, model_max)


cross_model = CrossEncoder(model_name, max_length=chosen_len+20)  # +10 for safety margin

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## SEMANTIC MATCHING IN ACTION:
"""
This block performs semantic matching for `S` and `SE` processing modes.

A partial match threshold of 50 is defined first. The matched Snowflake results
are then prepared by filling missing input and ATOM titles with blank strings
and normalizing both titles using `normalize_title`.

The script generates embeddings for the normalized input titles and ATOM titles
using the SentenceTransformer model. It then calculates row-wise cosine
similarity between each input title and its corresponding ATOM title. The score
is converted into a percentage and stored as `SEMANTIC_SCORE`.

For borderline matches, the CrossEncoder model is applied to generate a
`CROSS_SCORE`. The final match decision is then calculated using
`final_decision`, which considers semantic score, cross score, title numbers,
AKA logic, and extra-word rules.

After scoring, the script:
    - Rounds semantic scores.
    - Calculates extra words between input and candidate titles.
    - Sets `FINAL_SCORE`.
    - Adds explainability columns such as confidence, reason, and evidence.
    - Drops temporary normalized title columns.

The scored results are then split into:
    - Top matches: non-rejected candidates selected using top-N logic.
    - Partial matches: candidates with semantic score above the partial threshold
    that were not already included in top matches.
    - No-match records: input titles that do not appear in either top or partial
    match outputs.

For top and partial matches, the script ranks candidates by ID, removes duplicate
node identifiers, maps original input titles back from `INPUT_TITLES_0`, selects
the required output columns, and prepares the final DataFrames for Excel output.
"""

PARTIAL_THRESHOLD = 50


if IP in ("S", "SE"):
    # ----------------------------
    # STANDALONE
    # ----------------------------

    Matches["INPUT_TITLE"] = Matches["INPUT_TITLE"].fillna("")
    Matches["ATOM_TITLE"]  = Matches["ATOM_TITLE"].fillna("")

    Matches["INPUT_TITLE_NORM"] = Matches["INPUT_TITLE"].apply(smart_title_matching_functions.normalize_title)
    Matches["ATOM_TITLE_NORM"]  = Matches["ATOM_TITLE"].apply(smart_title_matching_functions.normalize_title)

    # Semantic similarity
    def rowwise_cosine(a, b):
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
        return (a * b).sum(dim=1)

    colA_emb = model.encode(Matches["INPUT_TITLE_NORM"].tolist(), convert_to_tensor=True)
    colB_emb = model.encode(Matches["ATOM_TITLE_NORM"].tolist(),  convert_to_tensor=True)

    pair_scores = rowwise_cosine(colA_emb, colB_emb)
    Matches["SEMANTIC_SCORE"] = (pair_scores.cpu().numpy() * 100)

    # Cross encoder
    Matches = smart_title_matching_functions.apply_cross_encoder(
        Matches, "INPUT_TITLE_NORM", "ATOM_TITLE_NORM",
        "SEMANTIC_SCORE", "CROSS_SCORE", cross_model
    )

    # Final decision
    Matches["MATCH_RESULT"] = Matches.apply(
        lambda r: smart_title_matching_functions.final_decision(
            r["INPUT_TITLE_NORM"], r["ATOM_TITLE_NORM"],
            r["SEMANTIC_SCORE"], r["CROSS_SCORE"]
        ),
        axis=1
    )

    Matches["SEMANTIC_SCORE"] = Matches["SEMANTIC_SCORE"].round(1)
    Matches["EXTRA_WORDS"] = Matches.apply(
        lambda row: smart_title_matching_functions.extra_word_ratio(row["INPUT_TITLE_NORM"], row["ATOM_TITLE_NORM"]),
        axis=1
    )
    Matches["FINAL_SCORE"] = Matches["SEMANTIC_SCORE"]

    Matches = smart_title_matching_functions.add_explainability_columns(
        Matches,
        "FINAL_SCORE",
        "MATCH_RESULT"
    )
    Matches = Matches.drop(columns=["INPUT_TITLE_NORM", "ATOM_TITLE_NORM"], errors="ignore")

    # Keep scored universe (avoid full .copy() unless you need to mutate independently)
    Matches_scored_all = Matches

    # -------- Top match (Top3, non-reject) --------
    Matches_valid = Matches_scored_all[Matches_scored_all["MATCH_RESULT"] != "Reject"]

    Matches_top3 = smart_title_matching_functions.top_n_per_id(Matches_valid, "ID", "SEMANTIC_SCORE", 3)
    Matches_top3 = smart_title_matching_functions.add_match_ranking(
        Matches_top3,
        "ID",
        "SEMANTIC_SCORE"
    )


    # Convert MPM_NUMBER safely (fast + stable)
    if "MPM_NUMBER" in Matches_top3.columns:
        Matches_top3["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Matches_top3["MPM_NUMBER"]]

    # Dedupe after shrinking
    if "NODE_IDENTIFIER" in Matches_top3.columns:
        Matches_top3 = Matches_top3.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    # Add INPUT title/features using map (faster than merge)
    features_map = INPUT_TITLES_0.set_index("Sno")["Features"]
    Matches_top3["INPUT_TITLES"] = Matches_top3["ID"].map(features_map)

    # Select final columns (only existing)
    if IP == "SE":
        desired_cols_top = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER","LIBRARY_TITLE_FULL", "MPM_NUMBER", "SEMANTIC_SCORE", "MATCH_RESULT",
            "CONFIDENCE",
            "FINAL_REASON",
            "MATCH_RANK",
            "KEEP_RECOMMENDED",
            "MATCH_EVIDENCE",
            "CHILDREN_STATUS",
            "PI_UUID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    else:
        desired_cols_top = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER","LIBRARY_TITLE_FULL", "MPM_NUMBER", "PARENT_TITLE","PARENT_ENTITY","PARENT_MPM",
            "SEMANTIC_SCORE", "MATCH_RESULT",
            "CONFIDENCE",
            "FINAL_REASON",
            "MATCH_RANK",
            "KEEP_RECOMMENDED",
            "MATCH_EVIDENCE",
            "PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    Matches_top3 = Matches_top3[[c for c in desired_cols_top if c in Matches_top3.columns]]

    top_ids = set(Matches_top3["ID"].dropna().unique())

    # -------- Partial match (Top4 per ID, score>=threshold, excluding top IDs) --------
    Partial_pool = Matches_scored_all[
        (Matches_scored_all["SEMANTIC_SCORE"] >= PARTIAL_THRESHOLD) &
        (~Matches_scored_all["ID"].isin(top_ids))
    ]

    # If you want to exclude rejects in partial too, uncomment:
    # Partial_pool = Partial_pool[Partial_pool["MATCH_RESULT"] != "Reject"]

    Partial_matches = smart_title_matching_functions.top_n_per_id(Partial_pool, "ID", "SEMANTIC_SCORE", 4)
    Partial_matches = smart_title_matching_functions.add_match_ranking(
        Partial_matches,
        "ID",
        "SEMANTIC_SCORE"
    )

    if "MPM_NUMBER" in Partial_matches.columns:
        Partial_matches["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Partial_matches["MPM_NUMBER"]]

    if "NODE_IDENTIFIER" in Partial_matches.columns:
        Partial_matches = Partial_matches.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    Partial_matches["INPUT_TITLES"] = Partial_matches["ID"].map(features_map)
    if IP == "SE":
        desired_cols_partial = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE","LIBRARY_TITLE_FULL",
            "NODE_IDENTIFIER", "MPM_NUMBER", "SEMANTIC_SCORE", "MATCH_RESULT",
            "CONFIDENCE",
            "FINAL_SCORE",
            "FINAL_REASON",
            "MATCH_RANK",
            "KEEP_RECOMMENDED",
            "MATCH_EVIDENCE",
            "CHILDREN_STATUS",
            "PI_UUID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    else:
        desired_cols_partial = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE","LIBRARY_TITLE_FULL",
            "NODE_IDENTIFIER", "MPM_NUMBER", "PARENT_TITLE","PARENT_ENTITY","PARENT_MPM",
            "SEMANTIC_SCORE", "MATCH_RESULT",
            "CONFIDENCE",
            "FINAL_SCORE",
            "FINAL_REASON",
            "MATCH_RANK",
            "KEEP_RECOMMENDED",
            "MATCH_EVIDENCE",
            "PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
            
        ]
    Partial_matches = Partial_matches[[c for c in desired_cols_partial if c in Partial_matches.columns]]
    perfects=Partial_matches[(Partial_matches["MATCH_RESULT"]=="Perfect Match")|(Partial_matches["MATCH_RESULT"]=="Possible Match")|(Partial_matches["SEMANTIC_SCORE"]>=85)]
    perfects["MATCH_RESULT"]=["Possible Match" if i=="Reject" else i for i in perfects["MATCH_RESULT"]]
    Matches_top3=pd.concat([Matches_top3,perfects])
    Matches_top3 = Matches_top3.sort_values(
    ["ID","MATCH_RANK"],
    ascending=[True,True])
    top_ids = set(Matches_top3["ID"].dropna().unique())
    Partial_matches = Partial_matches[~Partial_matches["ID"].isin(perfects["ID"].unique())]
    Partial_matches = Partial_matches.sort_values(
    ["ID","MATCH_RANK"],
    ascending=[True,True])
    partial_ids = set(Partial_matches["ID"].dropna().unique())
    Matches_top3 = Matches_top3.drop(
        columns=["MATCH_RESULT", "KEEP_RECOMMENDED","FINAL_SCORE"],
        errors="ignore"
    )
    Partial_matches = Partial_matches.drop(
        columns=["MATCH_RESULT", "KEEP_RECOMMENDED","FINAL_SCORE"],
        errors="ignore"
    )

    # -------- No match --------
    all_input_ids = set(INPUT_TITLES_0["Sno"].dropna().unique())
    no_match_ids = sorted(INPUT_TITLES_0[~INPUT_TITLES_0["Sno"].isin(top_ids.union(partial_ids))]["Sno"])

    No_match_sheet = (
        INPUT_TITLES_0[INPUT_TITLES_0["Sno"].isin(no_match_ids)][["Sno", "Features","Years"]]
        .rename(columns={"Sno": "ID", "Features": "INPUT_TITLE"})
        .sort_values("ID")
        .reset_index(drop=True)
    )

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
    """
    This block handles the `ALL` processing flow, where Series candidates are scored
    first and reviewed by the user before Season and Episode candidates are scored.

    The script starts by scoring Series-level Snowflake candidates using semantic
    similarity and CrossEncoder logic. Non-rejected Series matches are filtered,
    ranked, deduplicated by ID and NODE_IDENTIFIER, and written to a formatted
    `Series_Selection.xlsx` file.

    The user is asked to review the Series selection file and choose `Yes` or `No`
    in the `Select_Series` column. After the user saves the file, the script reads
    the selected rows and captures the selected Series `NODE_IDENTIFIER` values.

    The selected Series rows are treated as final Series-level matches. Season and
    Episode candidate records are then filtered based on the selected Series parent
    identifiers. This ensures that Season and Episode matching is scoped to the
    Series selected by the user.

    Next, the script scores Season and Episode candidates using child-match scoring
    logic. This compares both parent or series titles and child or episode titles,
    then applies combined match logic to produce the final decision.

    The selected Series, scored Season, and scored Episode results are combined into
    one match DataFrame. The script then:
        - Converts IDs to nullable integers.
        - Keeps non-rejected records as the valid match pool.
        - Selects top matches using top-N logic.
        - Ranks matches within each ID.
        - Removes duplicate NODE_IDENTIFIER values.
        - Builds partial matches using the partial score threshold.
        - Promotes strong partial matches into the top-match output.
        - Creates a no-match sheet for input IDs not found in top or partial matches.
        - Reorders final output columns for Excel reporting.

    The final outputs from this block are:
        - `Matches_top3` for top Series, Season, and Episode matches.
        - `Partial_matches` for lower-confidence but reviewable matches.
        - `No_match_sheet` for input records with no suitable match.
    """

else:
    # ----------------------------
    # ALL: SERIES FIRST, USER SELECTION, THEN SEASON + EPISODE
    # ----------------------------

    logger.info("Processing ALL flow: Series first, then Season/Episode based on user selection...")

    # 1. Score Series candidates first
    Series_scored_all = smart_title_matching_functions.score_series_matches(
    Series_SnowFlake_Results,
    INPUT_TITLES_0,
    model,
    cross_model)

    if Series_scored_all.empty or "FINAL_MATCH_RESULT" not in Series_scored_all.columns:
        Series_top3_for_selection = pd.DataFrame(
            columns=[
                "ID",
                "INPUT_TITLE",
                "ATOM_TITLE",
                "LIBRARY_TITLE_FULL",
                "INPUT_YEAR",
                "YEARS",
                "TT_CODES",
                "IP_TYPE",
                "NODE_IDENTIFIER",
                "MPM_NUMBER",
                "FINAL_SCORE",
                "FINAL_MATCH_RESULT",
                "CHILDREN_STATUS",
                "Select_Series"
            ]
        )
    else:
        Series_valid = Series_scored_all[
            Series_scored_all["FINAL_MATCH_RESULT"] != "Reject"
        ].copy()

        Series_top3_for_selection = smart_title_matching_functions.top_n_per_id(
            Series_valid,
            "ID",
            "FINAL_SCORE",
            3
        )


    if "NODE_IDENTIFIER" in Series_top3_for_selection.columns:
        Series_top3_for_selection = Series_top3_for_selection.drop_duplicates(
            subset=["ID", "NODE_IDENTIFIER"],
            keep="first"
        )

    series_selection_cols = [
        "ID",
        "INPUT_TITLE",
        "ATOM_TITLE",
        "INPUT_YEAR",
        "YEARS",
        "TT_CODES",
        "IP_TYPE",
        "LIBRARY_TITLE_FULL",
        "NODE_IDENTIFIER",
        "MPM_NUMBER",
        "CONFIDENCE",
        "FINAL_SCORE",
        "FINAL_REASON",
        "MATCH_RANK",
        "KEEP_RECOMMENDED",
        "MATCH_EVIDENCE",
        "CHILDREN_STATUS",
        "PI_UUID",
        "PROPERTY_ID",
        "HBO_ID",
        "META_ID",
        "TURNER_TITLEID",
        "MMS3_MCODE",
        "DASH_TITLE_ID",
        "ALEPH_ID",
        "IBROADCAST_EMEA_ID",
        "IBROADCAST_APAC_ID"
    ]

    Series_top3_for_selection = Series_top3_for_selection[
        [c for c in series_selection_cols if c in Series_top3_for_selection.columns]
    ].copy()

    Series_top3_for_selection["Select_Series"] = "No"

    series_selection_path = os.path.join(Output_folder, "Series_Selection.xlsx")

    smart_title_matching_functions.write_series_selection_file(
        Series_top3_for_selection,
        series_selection_path
    )

    logger.info(f"Series selection file created: {series_selection_path}")

    print("\nSeries selection file created:")
    print(series_selection_path)
    print("\nPlease open the file, select Yes/No in Select_Series column, save and close Excel.")
    print("After saving the file, come back here and press Enter.\n")

    smart_title_matching_functions.open_excel_file(series_selection_path)

    input("Press Enter after you saved the Series_Selection.xlsx file...")

    # 2. Read user selected Series file
    selected_series_file = pd.read_excel(
    series_selection_path,
    sheet_name="Series match",
    header=1)

    if "Select_Series" not in selected_series_file.columns:
        selected_series_file["Select_Series"] = "No"

    if "NODE_IDENTIFIER" not in selected_series_file.columns:
        selected_series_file["NODE_IDENTIFIER"] = np.nan

    selected_series_file["Select_Series"] = (
        selected_series_file["Select_Series"]
        .fillna("No")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    selected_series = selected_series_file[
        selected_series_file["Select_Series"].eq("YES")
    ].copy()

    selected_parent_ids = selected_series["NODE_IDENTIFIER"].dropna().astype(str).unique().tolist()

    logger.info(f"Selected Series NODE_IDENTIFIER count: {len(selected_parent_ids)}")

    # 3. Final selected series output
    selected_series_output = selected_series.copy()
    selected_series_output["MATCH_LEVEL"] = "Series"

    # Keep Yes-selected rows as Top matches
    selected_series_output["FINAL_SCORE"] = selected_series_output["FINAL_SCORE"].round(1)

    # 4. Filter Season/Episode based on selected Series NODE_IDENTIFIER
    Season_candidates = smart_title_matching_functions.filter_children_by_selected_series(
        Season_SnowFlake_Results,
        selected_parent_ids
    )

    Episode_candidates = smart_title_matching_functions.filter_children_by_selected_series(
        Episode_SnowFlake_Results,
        selected_parent_ids
    )

    Season_candidates["MATCH_LEVEL"] = "Season"
    Episode_candidates["MATCH_LEVEL"] = "Episode"

    # 5. Score Season and Episode candidates
    Season_scored_all = smart_title_matching_functions.score_child_matches(
        Season_candidates,
        INPUT_TITLES_0,
        model,
        cross_model
    )

    Episode_scored_all = smart_title_matching_functions.score_child_matches(
        Episode_candidates,
        INPUT_TITLES_0,
        model,
        cross_model
    )

    # 6. Combine Series + Season + Episode scored results
    Matches_scored_all = pd.concat(
        [
            selected_series_output,
            Season_scored_all,
            Episode_scored_all
        ],
        ignore_index=True,
        sort=False
    )

    Matches_scored_all["ID"] = pd.to_numeric(
        Matches_scored_all["ID"],
        errors="coerce"
    ).astype("Int64")

    # 7. Top match
    Matches_valid = Matches_scored_all[
        Matches_scored_all["FINAL_MATCH_RESULT"] != "Reject"
    ].copy()

    Matches_top3 = smart_title_matching_functions.top_n_per_id(
        Matches_valid,
        "ID",
        "FINAL_SCORE",
        3
    )
    Matches_top3 = smart_title_matching_functions.add_match_ranking(
        Matches_top3,
        "ID",
        "FINAL_SCORE"
    )
    if "NODE_IDENTIFIER" in Matches_top3.columns:
        Matches_top3 = Matches_top3.drop_duplicates(
            subset=["ID", "NODE_IDENTIFIER"],
            keep="first"
        )

    # 8. Partial match
    top_ids = set(Matches_top3["ID"].dropna().unique())

    Partial_pool = Matches_scored_all[
        (Matches_scored_all["FINAL_SCORE"] >= PARTIAL_THRESHOLD)
        & (~Matches_scored_all["ID"].isin(top_ids))
    ].copy()

    Partial_matches = smart_title_matching_functions.top_n_per_id(
        Partial_pool,
        "ID",
        "FINAL_SCORE",
        4
    )
    Partial_matches = smart_title_matching_functions.add_match_ranking(
        Partial_matches,
        "ID",
        "FINAL_SCORE"
    )
    if "NODE_IDENTIFIER" in Partial_matches.columns:
        Partial_matches = Partial_matches.drop_duplicates(
            subset=["ID", "NODE_IDENTIFIER"],
            keep="first"
        )

    perfects = Partial_matches[
        (Partial_matches["FINAL_MATCH_RESULT"].eq("Perfect Match"))
        | (
            Partial_matches["FINAL_MATCH_RESULT"].eq("Possible Match")
            & (Partial_matches["FINAL_SCORE"] >= 85)
        )
    ].copy()

    perfects["FINAL_MATCH_RESULT"] = [
        "Possible Match" if i == "Reject" else i
        for i in perfects["FINAL_MATCH_RESULT"]
    ]

    Matches_top3 = pd.concat([Matches_top3, perfects], ignore_index=True, sort=False)
    #Matches_top3 = Matches_top3.sort_values(by="PARENT_TITLE")
    Matches_top3 = Matches_top3.sort_values(
                                        ["SERIES_TITLE","ID","MATCH_RANK"],
                                        ascending=[True,True,True]
                                    )
    Partial_matches = Partial_matches[
        ~Partial_matches["ID"].isin(perfects["ID"].unique())
    ].copy()
    #Partial_matches= Partial_matches.sort_values(by="PARENT_TITLE")
    Partial_matches = Partial_matches.sort_values(
    ["SERIES_TITLE","ID","MATCH_RANK"],
    ascending=[True,True,True])
    top_ids = set(Matches_top3["ID"].dropna().unique())
    partial_ids = set(Partial_matches["ID"].dropna().unique())

    # 9. No match
    no_match_ids = sorted(
        INPUT_TITLES_0[
            ~INPUT_TITLES_0["Sno"].isin(top_ids.union(partial_ids))
        ]["Sno"]
    )

    no_match_cols = [
        c for c in [
            "Sno",
            "IP_TYPE",
            "SERIES_TITLE",
            "SeasonNumber",
            "EpisodeNumber",
            "Episode",
            "EPISODE_TITLE",
            "Years"
        ]
        if c in INPUT_TITLES_0.columns
    ]

    No_match_sheet = INPUT_TITLES_0[
        INPUT_TITLES_0["Sno"].isin(no_match_ids)
    ][no_match_cols].copy()

    No_match_sheet = No_match_sheet.sort_values(by="SERIES_TITLE")

    No_match_sheet = No_match_sheet.rename(
        columns={
            "Sno": "ID",
            "Episode": "INPUT_TITLE",
            "EPISODE_TITLE": "INPUT_TITLE"
        }
    )

    No_match_sheet = No_match_sheet.sort_values("ID").reset_index(drop=True)

    # 10. Final column arrangement
    desired_cols_all = [
    "ID",
    "INPUT_TITLE",
    "SERIES_TITLE",
    "PARENT_TITLE",
    "ATOM_TITLE",
    "INPUT_YEAR",
    "YEARS",
    "TT_CODES",

    "IP_TYPE",
    "NODE_IDENTIFIER",
    "LIBRARY_TITLE_FULL",

    "PARENT_ENTITY",
    "PARENT_MPM",
    "CHILDREN_STATUS",

    "FINAL_SCORE",
    "MATCH_LEVEL",
    "CONFIDENCE",
    "FINAL_REASON",
    "MATCH_RANK",
    "MATCH_EVIDENCE",

    "MPM_NUMBER",
    "MPM_PRODUCT_NUMBER",
    "PI_UUID",
    "PROPERTY_ID",
    "HBO_ID",
    "META_ID",
    "TURNER_TITLEID",
    "MMS3_MCODE",
    "DASH_TITLE_ID",
    "ALEPH_ID",
    "IBROADCAST_EMEA_ID",
    "IBROADCAST_APAC_ID"]

    Matches_top3 = Matches_top3[
        [c for c in desired_cols_all if c in Matches_top3.columns]
    ].copy()

    Partial_matches = Partial_matches[
        [c for c in desired_cols_all if c in Partial_matches.columns]
    ].copy()

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# -----------------------------
# Export
# -----------------------------

"""
This block exports the final matching results into a formatted Excel workbook.

The script creates an Excel file named `Output.xlsx` inside the selected output
folder using the `xlsxwriter` engine. The Excel writer is passed into
`set_writer`, which initializes the workbook object and reusable formatting
rules.

The script then determines the Excel formatting mode based on the selected IP
category. This mode controls which grouped column layout should be used when
writing the output sheets.

The final workbook is written in the following sheet order:
    1. Top match
    2. Partial match
    3. No match

The `Top match` and `Partial match` sheets are written using
`apply_grouped_formatting`, which applies grouped headers, column ordering,
identifier formatting, hyperlinks, ingestion indicators, and warning highlights.

The `No match` sheet is written using `format_no_match`, which applies simpler
header, width, numeric cleanup, and border formatting for unmatched records.

After the workbook is saved and closed, the script logs that the final output is
ready.

"""
with pd.ExcelWriter(Output_folder + "/Output.xlsx", engine="xlsxwriter") as writer:
    smart_title_matching_functions.set_writer(writer)

    workbook = writer.book

    # Decide mode (as in your code)
    ip_mode_map = {
    "S": "STANDALONE",
    "SEA": "SEASON",
    "SE": "SERIES",
    "E": "EPISODICS",
    "ALL": "EPISODICS"
    }
    ip_mode = ip_mode_map.get(IP, "STANDALONE")


    # SHEET ORDER: Top match -> Partial match -> No match
    smart_title_matching_functions.apply_grouped_formatting(Matches_top3, "Top match", ip_mode)
    smart_title_matching_functions.apply_grouped_formatting(Partial_matches, "Partial match", ip_mode)
    smart_title_matching_functions.format_no_match(No_match_sheet, "No match")
    

logger.info("Final output is ready...")

c:\Users\mshanmugam\OneDrive - Warner Bros. Discovery\JUPYTER_PY\AI Projects\Smart Title Matching (AI-Based)\smart_title_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Packages imported successfully


 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJbc9owEIX%2Fikd9tiVzC2iAjBsmAzPQ0GBI2jdhL6BiS0YrY9JfX5lLJ31IZvqmkc7qO7tn%2B%2FenPPOOYFBqNSBhwIgHKtGpVNsBWcaPfpd4aIVKRaYVDMgbILkf9lHkWcGj0u7UMxxKQOu5jxTy%2BmFASqO4FiiRK5EDcpvwRTSb8kbAuEAEYx2OXEtSlI61s7bglFZVFVTNQJstbTDGKOtRp6olX8g7RPE5ozDa6kRnt5KT6%2BkDREhZq0Y4hSPMr4VfpbqM4DPK%2BiJCPo7juT9%2FWsTEi27dPWiFZQ5mAeYoE1g%2BTy8G0Dl4mUTNDut0ghJ9EGj9MEClq00m9pDovCit%2BzZwJ7qBlGZ6K92wJqMBKfYynf%2BawfejTVvrfSheooWwVfdHFe9em9Fsvxvv2Op3eyxepwfXCPFWt2gbdbQTxBImqg7UuivW6Pis64ftOGzxdo83wiBs934Sb%2BQClUrYc%2BXNNaIOqnV69iWKgv61TOG0L1X3cLfNV0u70eaJne5qOa2jIpdt4We2Gf73DPr0ffl18765MCajuc5k8uY9apML%2B3FWrqvzjUz9zVnKIRcyi9LUAKLLLMt09WBAWLfg1pRA6PBC%2FXfFh38A&RelayState=ver%3A3-hint%3A2241

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 90c2079a-3dad-40b4-b8ee-284eb635d7ec)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


✅ Folder cleaned successfully
Selected Output Folder: C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Smart Title Matching (AI-Based)/Output


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 98d4d3be-6aff-4cff-b7c3-47dd2f3f418f)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: cb7925ae-4c8a-4e16-886c-850c130568f4)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-Min